In [1]:
import numpy as np
import pandas as pd

df = pd.read_csv("train.csv")

df.drop(columns=["PassengerId","Name","Ticket","Cabin"], inplace=True)

df.head()


,Survived,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,0,3,male,22.0,1,0,7.2500,S
1,1,1,female,38.0,1,0,71.2833,C
2,1,3,female,26.0,0,0,7.9250,S
3,1,1,female,35.0,1,0,53.1000,S
4,0,3,male,35.0,0,0,8.0500,S


In [2]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape, X_test.shape)
X_train.head()


(712, 7) (179, 7)


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [3]:
num_cols = ["Age", "Fare"]
cat_cols = ["Sex", "Embarked"]
passthrough_cols = ["Pclass", "SibSp", "Parch"]

print(num_cols, cat_cols, passthrough_cols)


['Age', 'Fare'] ['Sex', 'Embarked'] ['Pclass', 'SibSp', 'Parch']


In [4]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, num_cols),
    ("cat", categorical_transformer, cat_cols),
    ("pass", "passthrough", passthrough_cols)
])


In [5]:
from sklearn.tree import DecisionTreeClassifier

model = DecisionTreeClassifier(random_state=42)

pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", model)
])

pipe.fit(X_train, y_train)

print("✅ Pipeline trained!")


✅ Pipeline trained!


In [6]:
from sklearn.metrics import accuracy_score

y_pred = pipe.predict(X_test)

print("✅ Accuracy:", accuracy_score(y_test, y_pred))


✅ Accuracy: 0.7821229050279329


# make_pipeline

In [9]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 1) Load data
df = pd.read_csv("train.csv")

# 2) Drop unwanted columns
df.drop(columns=["PassengerId", "Name", "Ticket", "Cabin"], inplace=True)

# 3) Split X and y
X = df.drop(columns=["Survived"])
y = df["Survived"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

# 4) Columns
num_cols = ["Age", "Fare"]
cat_cols = ["Sex", "Embarked"]
pass_cols = ["Pclass", "SibSp", "Parch"]

# 5) make_pipeline transformers
numeric_transformer = make_pipeline(
    SimpleImputer(strategy="median")
)

categorical_transformer = make_pipeline(
    SimpleImputer(strategy="most_frequent"),
    OneHotEncoder(handle_unknown="ignore")
)

# 6) ColumnTransformer
preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
        ("pass", "passthrough", pass_cols)
    ]
)

# 7) Full pipeline = preprocessing + model
model = DecisionTreeClassifier(random_state=42)

pipe = make_pipeline(
    preprocessor,
    model
)

# 8) Train
pipe.fit(X_train, y_train)

# 9) Predict + Accuracy
y_pred = pipe.predict(X_test)
print("✅ Accuracy:", accuracy_score(y_test, y_pred))



✅ Accuracy: 0.7821229050279329
